# 05 · 文字條件：一句話如何導引生成

到目前為止,我們的迷你擴散模型只會**隨機**生數字——沒辦法叫它「生一個 7」。真正的 Stable Diffusion 能聽懂「一隻在沙灘上的柯基」,靠的是**文字條件(text conditioning)**。

## 兩個關鍵零件

> 1. **文字編碼器(CLIP)**:把一句 prompt 變成一個向量。CLIP 的神奇之處是它讓**文字向量和影像向量活在同一個空間**——「貓」的文字向量,會靠近貓的圖片向量。
> 2. **交叉注意力(cross-attention)**:在 U-Net 去噪的每一步,把這個文字向量「注入」進去,引導生成往 prompt 描述的方向走。

這課聚焦第一個零件,親手見證 **CLIP 把文字與影像對齊**——這是文字生圖的地基。

## 1. 安裝

In [ ]:
%pip install -q -U transformers

## 2. 載入 CLIP

CLIP 同時有「看圖」和「讀字」兩個編碼器,輸出可以直接比較的向量。

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP 載入完成。")

## 3. 文字與影像在同一個空間

給一張圖、幾句描述,CLIP 會告訴你哪句最配——證明文字與影像被對齊到同一空間。

In [ ]:
import requests
from PIL import Image

url = "https://raw.githubusercontent.com/jacobgil/pytorch-grad-cam/master/examples/both.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
captions = ["a photo of a cat", "a photo of a dog", "a photo of a car"]

inputs = proc(text=captions, images=image, return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    logits = clip(**inputs).logits_per_image.softmax(dim=1)[0]
for cap, p in zip(captions, logits):
    print(f"{cap:24s} {p.item():.1%}")
# CLIP 不需訓練就能說出圖裡有什麼——文字與影像確實對齊了。

## 4. 這跟生成有什麼關係?

文字生圖時,prompt 先被 CLIP 編碼成向量,再透過 cross-attention 注入 U-Net 的每一步去噪——讓「往哪去噪」被這句話牽引。我們手刻的去噪器只差這一塊;下一課直接用 diffusers,看完整的文字生圖跑起來。

In [ ]:
print("文字生圖 = 你手刻的去噪過程(02–04) + 文字向量透過 cross-attention 的引導。")
print("下一課:用 diffusers 一次看完整版。")

## 小結

- **文字條件**靠兩塊:CLIP(把 prompt 變向量)+ cross-attention(把向量注入去噪)。
- **CLIP 讓文字與影像活在同一向量空間**:不需微調就能判斷「圖裡是什麼」。
- 你前四課手刻的去噪器,加上這個文字引導,就是 Stable Diffusion 的骨架。

下一課:用 **diffusers** 跑真正的 **Stable Diffusion**,打一句 prompt 生一張圖。